<a href="https://colab.research.google.com/github/Hackathon-05-06-2026/Hackathon_Files/blob/main/Group_R__xgboost_svm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [138]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Machine Learning Libraries
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.metrics import accuracy_score, precision_score,recall_score,f1_score,confusion_matrix,classification_report
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV
from sklearn.svm import SVR
import xgboost as xgb
from xgboost import XGBClassifier
from xgboost import plot_importance

# Deep Learning Libraries
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau


plt.style.use('seaborn-v0_8-darkgrid')

In [129]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [130]:
df_train = pd.read_csv('/content/drive/MyDrive/super_dataset_C_train.csv')
print('Training data loaded successfully.')
display(df_train.head())

Training data loaded successfully.


,covid_concern,covid_knowledge,behavioral_antiviral_meds,behavioral_avoidance,behavioral_face_mask,behavioral_wash_hands,behavioral_large_gatherings,behavioral_outside_home,behavioral_touch_face,chronic_med_condition,...,employment_sector_mining,employment_sector_real_estate,employment_sector_retail,employment_sector_science,employment_sector_services,employment_sector_technology,employment_sector_transportation,employment_sector_utilities,employment_sector_wholesale,covid_vaccine
0,1.498120,-0.480873,-0.237211,0.604394,-0.280819,0.438429,-0.74467,1.428554,0.664463,1.529637,...,-0.138094,-0.134087,-0.089746,-0.276155,-0.098825,-0.100972,-0.107166,-0.177961,-0.209519,0
1,0.390024,-0.480873,-0.237211,0.604394,-0.280819,0.438429,-0.74467,-0.701023,0.664463,-0.672359,...,-0.138094,-0.134087,-0.089746,-0.276155,-0.098825,-0.100972,-0.107166,-0.177961,-0.209519,1
2,-1.826169,1.145981,-0.237211,-1.662560,-0.280819,-2.285143,-0.74467,-0.701023,-1.508681,1.529637,...,-0.138094,-0.134087,-0.089746,-0.276155,-0.098825,-0.100972,-0.107166,-0.177961,4.772840,0
3,0.390024,1.145981,-0.237211,0.604394,-0.280819,0.438429,-0.74467,-0.701023,0.664463,1.529637,...,-0.138094,-0.134087,-0.089746,-0.276155,-0.098825,-0.100972,-0.107166,-0.177961,-0.209519,1
4,-0.718072,-0.480873,-0.237211,0.604394,-0.280819,-2.285143,-0.74467,-0.701023,-1.508681,-0.672359,...,-0.138094,-0.134087,-0.089746,-0.276155,-0.098825,-0.100972,-0.107166,-0.177961,4.772840,0


In [131]:
print('DataFrame Info:')
df_train.info()

print('\nMissing values per column:')
print(df_train.isnull().sum())

DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4756 entries, 0 to 4755
Data columns (total 65 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   covid_concern                         4756 non-null   float64
 1   covid_knowledge                       4756 non-null   float64
 2   behavioral_antiviral_meds             4756 non-null   float64
 3   behavioral_avoidance                  4756 non-null   float64
 4   behavioral_face_mask                  4756 non-null   float64
 5   behavioral_wash_hands                 4756 non-null   float64
 6   behavioral_large_gatherings           4756 non-null   float64
 7   behavioral_outside_home               4756 non-null   float64
 8   behavioral_touch_face                 4756 non-null   float64
 9   chronic_med_condition                 4756 non-null   float64
 10  child_under_6_months                  4756 non-null   float64
 11  h

In [132]:
# Impute numerical columns with the median
for col in df_train.select_dtypes(include=['float64']).columns:
    if df_train[col].isnull().any():
        median_val = df_train[col].median()
        df_train[col].fillna(median_val, inplace=True)

# Impute categorical columns with the mode
for col in df_train.select_dtypes(include=['object']).columns:
    if df_train[col].isnull().any():
        mode_val = df_train[col].mode()[0] # .mode() can return multiple if ties, take the first
        df_train[col].fillna(mode_val, inplace=True)

print('Missing values after imputation:')
print(df_train.isnull().sum())

Missing values after imputation:
covid_concern                       0
covid_knowledge                     0
behavioral_antiviral_meds           0
behavioral_avoidance                0
behavioral_face_mask                0
                                   ..
employment_sector_technology        0
employment_sector_transportation    0
employment_sector_utilities         0
employment_sector_wholesale         0
covid_vaccine                       0
Length: 65, dtype: int64


In [133]:
#splitting data into train and test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print('X_train shape:', X_train.shape)
print('X_test shape:', X_test.shape)
print('y_train shape:', y_train.shape)
print('y_test shape:', y_test.shape)

X_train shape: (3804, 58)
X_test shape: (952, 58)
y_train shape: (3804,)
y_test shape: (952,)


### Clean Feature Names for XGBoost

In [134]:
# Function to clean column names for XGBoost compatibility
def clean_col_names(df):
    cols = df.columns
    new_cols = []
    for col in cols:
        # Replace problematic characters with underscores
        new_col = col.replace('[', '_').replace(']', '_').replace('<', 'less_than_').replace(' ', '_').replace(':', '_')
        new_cols.append(new_col)
    df.columns = new_cols
    return df

# Apply cleaning to X_train and X_test
X_train = clean_col_names(X_train.copy())
X_test = clean_col_names(X_test.copy())

print('Cleaned X_train columns (first 5):', X_train.columns[:5].tolist(), '...')
print('Cleaned X_test columns (first 5):', X_test.columns[:5].tolist(), '...')


Cleaned X_train columns (first 5): ['covid_concern', 'covid_knowledge', 'behavioral_antiviral_meds', 'behavioral_avoidance', 'behavioral_face_mask'] ...
Cleaned X_test columns (first 5): ['covid_concern', 'covid_knowledge', 'behavioral_antiviral_meds', 'behavioral_avoidance', 'behavioral_face_mask'] ...


# XG BOOST

In [139]:
#XGBOOST Model
baseline_xgb = XGBClassifier(
    n_estimators=1000,
    early_stopping_rounds=30,
    random_state=5303,
    objective='binary:logistic',
    eval_metric='logloss'
)
baseline_xgb.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, average='weighted'))
print("Recall   :", recall_score(y_test, y_pred, average='weighted'))
print("F1 Score :", f1_score(y_test, y_pred, average='weighted'))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report")
print(classification_report(y_test, y_pred))

Accuracy : 0.773109243697479
Precision: 0.7659465343540341
Recall   : 0.773109243697479
F1 Score : 0.7613591086273392

Confusion Matrix
[[578  63]
 [153 158]]

Classification Report
              precision    recall  f1-score   support

           0       0.79      0.90      0.84       641
           1       0.71      0.51      0.59       311

    accuracy                           0.77       952
   macro avg       0.75      0.70      0.72       952
weighted avg       0.77      0.77      0.76       952



In [147]:
#hyper parameter tuning

xgb_clf = XGBClassifier(
    random_state=5303,
    eval_metric='logloss'
)

param_dist = {
    'n_estimators': [100, 200, 500, 1000],
    'max_depth': [3, 5, 7, 9],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'min_child_weight': [1, 3, 5],
    'gamma': [0, 0.1, 0.3, 0.5]
}

random_search = RandomizedSearchCV(
    estimator=xgb_clf,
    param_distributions=param_dist,
    n_iter=50,
    scoring='accuracy',
    cv=5,
    verbose=2,
    n_jobs=-1,
    random_state=5303
)

random_search.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

print("Best Parameters:")
print(random_search.best_params_)

print("Best CV Accuracy:")
print(random_search.best_score_)

# Get the best model from random search
best_xgb_model = random_search.best_estimator_

# Make predictions using the best model
y_pred_tuned = best_xgb_model.predict(X_test)

print("\nMetrics for Tuned XGBoost Model:")
print("Accuracy :", accuracy_score(y_test, y_pred_tuned))
print("Precision:", precision_score(y_test, y_pred_tuned, average='weighted'))
print("Recall   :", recall_score(y_test, y_pred_tuned, average='weighted'))
print("F1 Score :", f1_score(y_test, y_pred_tuned, average='weighted'))

print("\nConfusion Matrix (Tuned Model)")
print(confusion_matrix(y_test, y_pred_tuned))

print("\nClassification Report (Tuned Model)")
print(classification_report(y_test, y_pred_tuned))

Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best Parameters:
{'subsample': 0.6, 'n_estimators': 200, 'min_child_weight': 1, 'max_depth': 3, 'learning_rate': 0.05, 'gamma': 0.1, 'colsample_bytree': 0.6}
Best CV Accuracy:
0.800471678539318

Metrics for Tuned XGBoost Model:
Accuracy : 0.8025210084033614
Precision: 0.7980678294155584
Recall   : 0.8025210084033614
F1 Score : 0.79457601140062

Confusion Matrix (Tuned Model)
[[584  57]
 [131 180]]

Classification Report (Tuned Model)
              precision    recall  f1-score   support

           0       0.82      0.91      0.86       641
           1       0.76      0.58      0.66       311

    accuracy                           0.80       952
   macro avg       0.79      0.74      0.76       952
weighted avg       0.80      0.80      0.79       952



### Applying and Evaluating the Tuned XGBoost Model on Test Data

In [ ]:
# Get the best model from random search (assuming random_search has been run)
best_xgb_model = random_search.best_estimator_

# Make predictions using the best model on the test data
y_pred_tuned_test = best_xgb_model.predict(X_test)

print("Metrics for Tuned XGBoost Model on Test Data:")
print("Accuracy :", accuracy_score(y_test, y_pred_tuned_test))
print("Precision:", precision_score(y_test, y_pred_tuned_test, average='weighted'))
print("Recall   :", recall_score(y_test, y_pred_tuned_test, average='weighted'))
print("F1 Score :", f1_score(y_test, y_pred_tuned_test, average='weighted'))

print("\nConfusion Matrix (Tuned Model on Test Data)")
print(confusion_matrix(y_test, y_pred_tuned_test))

print("\nClassification Report (Tuned Model on Test Data)")
print(classification_report(y_test, y_pred_tuned_test))

In [151]:
# 1. Load the new test dataset
# Please confirm this filename is correct, if not, update it.
df_new_test_data = pd.read_csv('/content/drive/MyDrive/super_dataset_C_test.csv')
print('New test data loaded successfully. Shape:', df_new_test_data.shape)

# 2. Apply the same preprocessing steps as the training data
# Impute numerical columns with the median
for col in df_new_test_data.select_dtypes(include=['float64']).columns:
    if df_new_test_data[col].isnull().any():
        median_val = df_new_test_data[col].median()
        df_new_test_data[col].fillna(median_val, inplace=True)

# Impute categorical columns with the mode (if any exist, based on original df_train structure)
for col in df_new_test_data.select_dtypes(include=['object']).columns:
    if df_new_test_data[col].isnull().any():
        # For consistent imputation, you might want to use the mode from the *training* data
        # For now, using mode from test data, but be aware of potential data leakage if not careful.
        mode_val = df_new_test_data[col].mode()[0]
        df_new_test_data[col].fillna(mode_val, inplace=True)

# Separate features (X_new_test) from any potential target (if present, like 'covid_vaccine')
# Assuming 'covid_vaccine' is the target and should not be in features for prediction.
if 'covid_vaccine' in df_new_test_data.columns:
    X_new_test = df_new_test_data.drop('covid_vaccine', axis=1)
else:
    X_new_test = df_new_test_data.copy()

# Apply the same column cleaning function
X_new_test = clean_col_names(X_new_test.copy())

# Ensure columns in X_new_test match X_train columns
# This is crucial if one-hot encoding or feature selection was applied earlier
# Get columns from X_train (assuming X_train represents the final feature set structure)
train_cols = X_train.columns.tolist()

# Reindex X_new_test to match X_train columns
# Fill new columns with 0 if they don't exist in X_new_test
# Drop columns from X_new_test that are not in X_train
X_new_test_aligned = X_new_test.reindex(columns=train_cols, fill_value=0)

print('New test data preprocessed and columns aligned. Shape:', X_new_test_aligned.shape)

# 3. Make predictions using the best tuned model
# Assuming 'best_xgb_model' is available from the hyperparameter tuning step
if 'best_xgb_model' in locals():
    predictions = best_xgb_model.predict(X_new_test_aligned)
    print('Predictions made successfully.')
    # Display first few predictions
    print('\nFirst 5 predictions:', predictions[:5])
else:
    print("Error: 'best_xgb_model' not found. Please run the hyperparameter tuning cell first.")
    predictions = None

# 4. Save predictions to a CSV file
if predictions is not None:
    # Create a DataFrame for predictions
    predictions_df = pd.DataFrame(predictions, columns=['predicted_covid_vaccine'])

    # If 'respondent_id' exists in the original test data, you might want to include it
    if 'respondent_id' in df_new_test_data.columns:
        predictions_df.insert(0, 'respondent_id', df_new_test_data['respondent_id'])

    output_filename = '/content/drive/MyDrive/new_test_predictions.csv'
    predictions_df.to_csv(output_filename, index=False)
    print(f'Predictions saved to {output_filename}')
    display(predictions_df.head())


New test data loaded successfully. Shape: (4749, 65)
New test data preprocessed and columns aligned. Shape: (4749, 58)
Predictions made successfully.

First 5 predictions: [0 0 0 0 0]
Predictions saved to /content/drive/MyDrive/new_test_predictions.csv


,respondent_id,predicted_covid_vaccine
0,4757,0
1,4758,0
2,4759,0
3,4760,0
4,4761,0
